In [2]:
import pandas as pd


import numpy as np

In [3]:
# 1. Leer el CSV
votaciones = pd.read_csv('hcdn_votaciones_historico.csv')

# 2. Eliminar las columnas inútiles
votaciones = votaciones.drop(columns=['Unnamed: 0', '¿QUÉ DIJO?'])

# 3. Renombrar las columnas para no tener mayúsculas, espacios ni signos de interrogación
votaciones = votaciones.rename(columns={
    'DIPUTADO': 'diputado',
    'BLOQUE': 'bloque',
    'PROVINCIA': 'provincia',
    '¿CÓMO VOTÓ?': 'voto'
})

# 4. Convertir la fecha (que ahora es texto) a un formato de Fecha real (Datetime)
# Esto es vital para después poder filtrar por "Año" o "Mes" en el dashboard
votaciones['fecha_votacion'] = pd.to_datetime(votaciones['fecha_votacion'], format='%d/%m/%Y', errors='coerce')

# Vemos el resultado limpio
print(votaciones.head())
print("\nEstructura final del dataset:")
print(votaciones.info())

votaciones.head(8)

                            diputado                        bloque  \
0  ABDALA DE MATARAZZO, Norma Amanda    Frente Cívico por Santiago   
1                 ABRAHAM, Alejandro  Frente para la Victoria - PJ   
2                  AGUAD, Oscar Raúl          Unión Cívica Radical   
3               AGUILAR, Lino Walter            Compromiso Federal   
4             ALEGRE, Gilberto Oscar              Frente Renovador   

             provincia        voto  id_votacion  \
0  Santiago del Estero  AFIRMATIVO            1   
1              Mendoza  AFIRMATIVO            1   
2              Córdoba  AFIRMATIVO            1   
3             San Luis     AUSENTE            1   
4         Buenos Aires  AFIRMATIVO            1   

                                     titulo_proyecto fecha_votacion  
0  Régimen previsional especial de carácter excep...     2015-10-07  
1  Régimen previsional especial de carácter excep...     2015-10-07  
2  Régimen previsional especial de carácter excep...     2015-

,diputado,bloque,provincia,voto,id_votacion,titulo_proyecto,fecha_votacion
0,"ABDALA DE MATARAZZO, Norma Amanda",Frente Cívico por Santiago,Santiago del Estero,AFIRMATIVO,1,Régimen previsional especial de carácter excep...,2015-10-07
1,"ABRAHAM, Alejandro",Frente para la Victoria - PJ,Mendoza,AFIRMATIVO,1,Régimen previsional especial de carácter excep...,2015-10-07
2,"AGUAD, Oscar Raúl",Unión Cívica Radical,Córdoba,AFIRMATIVO,1,Régimen previsional especial de carácter excep...,2015-10-07
3,"AGUILAR, Lino Walter",Compromiso Federal,San Luis,AUSENTE,1,Régimen previsional especial de carácter excep...,2015-10-07
4,"ALEGRE, Gilberto Oscar",Frente Renovador,Buenos Aires,AFIRMATIVO,1,Régimen previsional especial de carácter excep...,2015-10-07
5,"ALFONSÍN, Ricardo",Unión Cívica Radical,Buenos Aires,AUSENTE,1,Régimen previsional especial de carácter excep...,2015-10-07
6,"ALONSO, Laura",Unión PRO,C.A.B.A.,AUSENTE,1,Régimen previsional especial de carácter excep...,2015-10-07
7,"ALONSO, María Luz",Frente para la Victoria - PJ,La Pampa,AFIRMATIVO,1,Régimen previsional especial de carácter excep...,2015-10-07


In [4]:
import re
import pandas as pd

# ============================================================
# PASO 1 — Eliminar ruido procedimental
# ============================================================
patrones_ruido = [
    r'APARTAMIENTO DE REGLAMENTO',
    r'Apartamiento de [Rr]eglamento',
    r'Apartamiento del [Rr]eglamento',
    r'A N U L A D A',
    r'Plan de Labor Parlamentaria',
    r'Conjunto de proyectos',
    r'Conjunto de varios',
    r'^\s*-\s*Votación',
    r'MOCIÓN SOLICITADA POR',
    r'Moción solicitada por',
]
regex_ruido = '|'.join(patrones_ruido)
df = votaciones[~votaciones['titulo_proyecto'].str.contains(
    regex_ruido, regex=True, na=False
)].copy()
print(f"[1] Registros originales:          {len(votaciones):,}")
print(f"[1] Registros tras eliminar ruido: {len(df):,}")
print(f"[1] Ruido eliminado:               {len(votaciones)-len(df):,}")

# ============================================================
# PASO 2 — Extraer título base + fecha de sesión
# ============================================================
def extraer_titulo_base(titulo):
    if pd.isna(titulo):
        return titulo
    titulo = re.sub(
        r'\s*-\s*(Artículos?\s*\d+[°º]?[^–]*|En General[^–]*|En Particular[^–]*|Votación en General[^–]*)',
        '', titulo, flags=re.IGNORECASE
    )
    titulo = re.sub(r'\s*\d{2}/\d{2}/\d{4}\s*-\s*\d{2}:\d{2}\s*$', '', titulo)
    return titulo.strip()

df['titulo_base'] = df['titulo_proyecto'].apply(extraer_titulo_base)
df['fecha_votacion'] = pd.to_datetime(df['fecha_votacion'])
df['fecha_base'] = df['fecha_votacion'].dt.date

# fecha_sesion = primer día del grupo (diputado, titulo_base)
# resuelve cruces de medianoche
df['fecha_sesion'] = df.groupby(
    ['diputado', 'titulo_base']
)['fecha_base'].transform('min')

print(f"\n[2] Títulos únicos originales: {df['titulo_proyecto'].nunique():,}")
print(f"[2] Títulos base únicos:        {df['titulo_base'].nunique():,}")

# ============================================================
# PASO 3 — Flag de votación "En General"
# ============================================================
df['es_voto_general'] = df['titulo_proyecto'].str.contains(
    r'En General|EN GENERAL', regex=True, na=False
)
print(f"\n[3] Registros con voto 'En General': {df['es_voto_general'].sum():,}")

# ============================================================
# PASO 4 — Consolidación vectorizada
# KEY incluye fecha_sesion para aislar sesiones distintas
# con mismo titulo_base (ej: "Votación en General y Particular...")
# ============================================================
def moda_voto(s):
    m = s.mode()
    return m.iloc[0] if len(m) > 0 else 'ABSTENCIÓN'

cols_contexto = ['bloque', 'provincia', 'fecha_votacion']
KEY = ['diputado', 'titulo_base', 'fecha_sesion']

# 4a — Grupos con voto "En General"
consolidado_general = (
    df[df['es_voto_general']]
    .groupby(KEY)
    .agg(
        voto=('voto', moda_voto),
        **{col: (col, 'first') for col in cols_contexto}
    )
    .reset_index()
    .assign(fuente_consolidacion='en_general')
)

# 4b — Grupos SIN voto "En General"
pares_con_general = set(
    zip(consolidado_general['diputado'],
        consolidado_general['titulo_base'],
        consolidado_general['fecha_sesion'])
)
df_art = df[~df['es_voto_general']].copy()
df_art['_key'] = list(zip(
    df_art['diputado'],
    df_art['titulo_base'],
    df_art['fecha_sesion']
))
df_solo_art = df_art[~df_art['_key'].isin(pares_con_general)].drop(columns='_key')

consolidado_articulos = (
    df_solo_art
    .groupby(KEY)
    .agg(
        voto=('voto', moda_voto),
        **{col: (col, 'first') for col in cols_contexto}
    )
    .reset_index()
    .assign(fuente_consolidacion='moda_articulos')
)

# 4c — Unir y limpiar
df_consolidado = (
    pd.concat([consolidado_general, consolidado_articulos], ignore_index=True)
    .drop(columns=['fecha_sesion'])
)

# ============================================================
# PASO 5 — Eliminar categorías de voto no informativas
# ============================================================
votos_ruido = ['PRESIDENTE', 'PENDIENTE DE INCORPORACIÓN', 'SIN VOTAR']
df_consolidado = df_consolidado[~df_consolidado['voto'].isin(votos_ruido)].copy()

# ============================================================
# VERIFICACIÓN FINAL
# ============================================================
print(f"\n[4] Registros originales:          {len(votaciones):,}")
print(f"[4] Tras eliminar ruido:            {len(df):,}")
print(f"[4] Tras consolidar:                {len(df_consolidado):,}")
print(f"[4] Reducción total:                {len(votaciones)-len(df_consolidado):,}")
print(f"\n[4] Fuente de consolidación:")
print(df_consolidado['fuente_consolidacion'].value_counts())
print(f"\n[4] Distribución del voto:")
print(df_consolidado['voto'].value_counts())
print(f"\n[4] Proyectos únicos: {df_consolidado['titulo_base'].nunique():,}")
print(f"[4] Diputados únicos: {df_consolidado['diputado'].nunique():,}")

[1] Registros originales:          578,507
[1] Registros tras eliminar ruido: 544,840
[1] Ruido eliminado:               33,667

[2] Títulos únicos originales: 2,107
[2] Títulos base únicos:        1,802

[3] Registros con voto 'En General': 7,196

[4] Registros originales:          578,507
[4] Tras eliminar ruido:            544,840
[4] Tras consolidar:                460,923
[4] Reducción total:                117,584

[4] Fuente de consolidación:
fuente_consolidacion
moda_articulos    454010
en_general          6913
Name: count, dtype: int64

[4] Distribución del voto:
voto
AFIRMATIVO    268458
AUSENTE       101332
NEGATIVO       80921
ABSTENCION     10212
Name: count, dtype: int64

[4] Proyectos únicos: 1,802
[4] Diputados únicos: 2,061


In [9]:
import unicodedata
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ============================================================
# PARÁMETROS
# ============================================================
THRESHOLD_FUZZY = 0.70
BATCH_SIZE      = 200
URL_PROYECTOS   = ('proyectos_parlamentarios2.1.csv')

# ============================================================
# PASO A — Cargar proyectos desde GitHub
# ============================================================
df_proyectos = pd.read_csv(URL_PROYECTOS)
cols_lookup  = ['TITULO', 'EXP_DIPUTADOS', 'EXP_SENADO', 'AUTOR', 'CAMARA_ORIGEN']
df_proy      = df_proyectos[cols_lookup].copy()
print(f"[A] Proyectos cargados: {len(df_proy):,}")

# ============================================================
# PASO B — Tabla de trabajo sobre titulo_base únicos
# ============================================================
titulos_unicos = (
    df_consolidado[['titulo_base']]
    .drop_duplicates()
    .reset_index(drop=True)
    .copy()
)
titulos_unicos['autor_final']   = None
titulos_unicos['camara_origen'] = None
titulos_unicos['fuente_autor']  = None
titulos_unicos['score_fuzzy']   = np.nan
print(f"[B] Títulos únicos en df_consolidado: {len(titulos_unicos):,}")

# ============================================================
# PASO C — Funciones de normalización de expedientes
# ============================================================
def normalizar_exp(exp):
    if pd.isna(exp):
        return None
    m = re.match(r'^(\d+)-([A-Z]+)-(\d{2,4})$', str(exp).strip().upper())
    if not m:
        return str(exp).strip().upper()
    num, tipo, año = m.groups()
    if len(año) == 2:
        año = f"19{año}" if int(año) >= 90 else f"20{año}"
    return f"{num}-{tipo}-{año}"

def extraer_exp_de_titulo(titulo):
    for pat in [
        r'(\d{1,5}-D-\d{2,4})',
        r'(\d{1,5}-S-\d{2,4})',
        r'(\d{1,5}-PE-\d{2,4})',
        r'(\d{1,5}-JGM-\d{2,4})',
        r'(\d{1,5}-CD-\d{2,4})',
    ]:
        m = re.search(pat, str(titulo), re.IGNORECASE)
        if m:
            return normalizar_exp(m.group(1))
    return None

# ============================================================
# PASO D — Match determinístico por expediente
# ============================================================
lookup_exp = {}
for _, row in df_proy.iterrows():
    for col in ['EXP_DIPUTADOS', 'EXP_SENADO']:
        exp = normalizar_exp(row[col])
        if exp and exp not in lookup_exp:
            lookup_exp[exp] = (row['AUTOR'], row['CAMARA_ORIGEN'])

titulos_unicos['exp_extraido'] = titulos_unicos['titulo_base'].apply(extraer_exp_de_titulo)

for idx in titulos_unicos[titulos_unicos['exp_extraido'].notna()].index:
    exp = titulos_unicos.at[idx, 'exp_extraido']
    if exp in lookup_exp:
        titulos_unicos.at[idx, 'autor_final']   = lookup_exp[exp][0]
        titulos_unicos.at[idx, 'camara_origen'] = lookup_exp[exp][1]
        titulos_unicos.at[idx, 'fuente_autor']  = 'determinístico'

n_det = (titulos_unicos['fuente_autor'] == 'determinístico').sum()
print(f"[D] Match determinístico: {n_det}/{len(titulos_unicos)} títulos únicos")

# ============================================================
# PASO E — Normalización de texto para TF-IDF
# ============================================================
def normalize_texto(text):
    if pd.isna(text):
        return ""
    text = str(text).upper()
    text = unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('ASCII')
    text = re.sub(r'\d{2}/\d{2}/\d{4}\s*-\s*\d{2}:\d{2}', '', text)
    text = re.sub(r'\d{1,5}-[A-Z]+-\d{2,4}', '', text)
    text = re.sub(r'[^A-Z0-9\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

# ============================================================
# PASO F — Match fuzzy TF-IDF para los sin autor
# ============================================================
df_fuzzy_input = titulos_unicos[titulos_unicos['fuente_autor'].isna()].copy()
print(f"[F] Títulos para fuzzy: {len(df_fuzzy_input)}")

proy_norm    = df_proy['TITULO'].apply(normalize_texto).tolist()
query_norm   = df_fuzzy_input['titulo_base'].apply(normalize_texto).tolist()

vectorizer   = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
tfidf        = vectorizer.fit_transform(proy_norm + query_norm)
proy_matrix  = tfidf[:len(proy_norm)]
query_matrix = tfidf[len(proy_norm):]

resultados_fuzzy = []
for i in range(0, len(df_fuzzy_input), BATCH_SIZE):
    batch        = query_matrix[i:i + BATCH_SIZE]
    scores       = cosine_similarity(batch, proy_matrix)
    best_indices = np.argmax(scores, axis=1)
    best_scores  = scores[np.arange(len(best_indices)), best_indices]
    for j, (best_idx, best_score) in enumerate(zip(best_indices, best_scores)):
        resultados_fuzzy.append({
            'idx_titulos':  df_fuzzy_input.index[i + j],
            'autor_final':  df_proy.iloc[best_idx]['AUTOR']         if best_score >= THRESHOLD_FUZZY else None,
            'camara_origen':df_proy.iloc[best_idx]['CAMARA_ORIGEN'] if best_score >= THRESHOLD_FUZZY else None,
            'fuente_autor': 'fuzzy'                                  if best_score >= THRESHOLD_FUZZY else None,
            'score_fuzzy':  round(float(best_score), 3),
        })

for r in resultados_fuzzy:
    idx = r['idx_titulos']
    titulos_unicos.at[idx, 'autor_final']   = r['autor_final']
    titulos_unicos.at[idx, 'camara_origen'] = r['camara_origen']
    titulos_unicos.at[idx, 'fuente_autor']  = r['fuente_autor']
    titulos_unicos.at[idx, 'score_fuzzy']   = r['score_fuzzy']

# ============================================================
# PASO G — Join de vuelta a df_consolidado
# ============================================================
df_consolidado = df_consolidado.merge(
    titulos_unicos[['titulo_base', 'autor_final', 'camara_origen',
                    'fuente_autor', 'score_fuzzy']],
    on='titulo_base',
    how='left'
)

# ============================================================
# REPORTE FINAL
# ============================================================
print(f"\n=== COBERTURA AUTOR en df_consolidado ===")
for fuente in ['determinístico', 'fuzzy', None]:
    mask = df_consolidado['fuente_autor'] == fuente if fuente else df_consolidado['fuente_autor'].isna()
    n_tit = df_consolidado[mask]['titulo_base'].nunique()
    n_reg = mask.sum()
    label = fuente if fuente else 'Sin autor'
    print(f"  {label:<20}: {n_tit:>5} títulos únicos | {n_reg:>8,} registros")

print(f"\n  TOTAL con autor: {df_consolidado['autor_final'].notna().sum():,} registros")
print(f"  TOTAL sin autor: {df_consolidado['autor_final'].isna().sum():,} registros")
print(f"\nColumnas en df_consolidado: {df_consolidado.columns.tolist()}")

[A] Proyectos cargados: 112,186
[B] Títulos únicos en df_consolidado: 1,802
[D] Match determinístico: 187/1802 títulos únicos
[F] Títulos para fuzzy: 1615

=== COBERTURA AUTOR en df_consolidado ===
  determinístico      :   187 títulos únicos |   47,793 registros
  fuzzy               :   216 títulos únicos |   55,233 registros
  Sin autor           :  1399 títulos únicos |  357,897 registros

  TOTAL con autor: 103,026 registros
  TOTAL sin autor: 357,897 registros

Columnas en df_consolidado: ['diputado', 'titulo_base', 'voto', 'bloque', 'provincia', 'fecha_votacion', 'fuente_consolidacion', 'autor_final', 'camara_origen', 'fuente_autor', 'score_fuzzy']


In [ ]:
df_consolidado.head(10)
df_consolidado.to_excel('df_consolidado.xlsx', index=False)


,diputado,titulo_base,voto,bloque,provincia,fecha_votacion,fuente_consolidacion,autor_final,camara_origen,fuente_autor,score_fuzzy
0,"ABALOVICH, Eduardo Antonio",Cesión del Estado Nacional al Estado Provincia...,ABSTENCION,Unión Cívica Radical,Santiago del Estero,1997-11-28,en_general,None,None,None,0.219
1,"ABALOVICH, Eduardo Antonio",Creación del Fondo Fiduciario Federal de Infra...,NEGATIVO,Unión Cívica Radical,Santiago del Estero,1997-05-07,en_general,None,None,None,0.605
2,"ABALOVICH, Eduardo Antonio",Distribución del producido del Impuesto sobre ...,ABSTENCION,Unión Cívica Radical,Santiago del Estero,1998-12-16,en_general,None,None,None,0.340
3,"ABALOVICH, Eduardo Antonio",Ley Nacional de la Actividad Nuclear,AUSENTE,Unión Cívica Radical,Santiago del Estero,1996-08-08,en_general,None,None,None,0.424
4,"ABALOVICH, Eduardo Antonio",Ley de persecución penal eficaz; Incorporación...,NEGATIVO,Unión Cívica Radical,Santiago del Estero,1997-08-06,en_general,None,None,None,0.385
5,"ABALOVICH, Eduardo Antonio",Moción del Diputado Storani solicitando la vue...,AFIRMATIVO,Unión Cívica Radical,Santiago del Estero,1997-08-06,en_general,None,None,None,0.237
6,"ABALOVICH, Eduardo Antonio",Modificación de la Ley 20.744 de Contrato de T...,AUSENTE,Unión Cívica Radical,Santiago del Estero,1996-09-04,en_general,None,None,None,0.447
7,"ABALOVICH, Eduardo Antonio",Modificación de las Leyes de Procedimiento Tri...,AUSENTE,Unión Cívica Radical,Santiago del Estero,1996-12-11,en_general,None,None,None,0.227
8,"ABALOVICH, Eduardo Antonio",Obligatoriedad de presentar los resultados de ...,AUSENTE,Unión Cívica Radical,Santiago del Estero,1996-12-12,en_general,None,None,None,0.117
9,"ABALOVICH, Eduardo Antonio",Presupuesto General de la Administración Nacio...,NEGATIVO,Unión Cívica Radical,Santiago del Estero,1996-11-20,en_general,"PEÑA, MARCOS",Diputados,fuzzy,0.800
